# M5.S5 · Guided integration lab
## Vector search: correct, parallel and reproducible

**About 45–60 minutes of active guided work; queue wait is additional.** This is Part 1 of M5S5.
The separate individual cluster assignment and group milestone have their own instructions.
The module quiz will be provided separately.

A retrieval service compares a query with stored vectors and returns the closest item.
We implement **exact top-1 cosine search in C**. The deterministic vectors represent an embedding
collection, but are synthetic: we do not load documents, train an embedding model, or evaluate semantic relevance.
This isolates the systems problem without downloads or package installations.

By the end you should be able to:
1. Check an exact search result against an independent small reference.
2. Explain MPI shards, OpenMP local work and a global result merge (M5S1–S2).
3. Run bounded comparisons through Slurm and interpret actual timings.
4. Explain what an existing Apptainer image can preserve and what remains host-dependent (M5S3).
5. Keep the code, configuration, evidence and limitations needed for a project handover (M5S4).

**Route:** tiny correctness check → OpenMP measurements → hybrid design → container rerun → evidence review.
The CPU route is required for the guided practical; hybrid and container execution depend on site availability.
No GPU is required. No successful cluster run is assumed.


## 0 · Run on SciTech

Open https://jupyter.iesci.tech:8085/ and choose **File → Open from URL**. Paste the raw notebook URL
provided with this session, choose Python 3 and save your own copy. This notebook writes into a new
student-owned directory and contains its own C source, runner and batch scripts.

The Python cells prepare files, check a tiny fixture and analyse reports. **Significant C workloads run
in separate Slurm allocations**, not on the login node or an ordinary Jupyter kernel.
If `sbatch` is unavailable in Jupyter, use the terminal instructions below in your cluster home directory.
The files must be on storage visible to both the notebook and compute nodes.

Use the [course cluster guide](https://github.com/OscarDiez/hpc_course/blob/main/cluster/README.md).
From outside campus: `ssh -J <username>@ssh.iesci.tech <username>@10.205.20.10`.
On the login node inspect `hostname`, `sinfo`, `module avail`, `module list`, `which gcc` and `which mpicc`.
Load the exact site modules in the generated **site_env.sh**; module names are deliberately not guessed.
No account/QoS is added. The scripts request one node, four CPU cores total, 1 GB and six minutes.

PASS = an executed check passed; FAIL = an attempted check failed; SKIPPED = an optional feature is
disabled/unavailable; NOT RUN = no execution evidence yet. A missing optional feature is not a benchmark result.


In [ ]:
from pathlib import Path
import csv, hashlib, json, math, os, shutil, struct, subprocess, sys, tempfile
from statistics import median
if 'WORK' not in globals():
    WORK = Path(tempfile.mkdtemp(prefix='m5s5_vector_', dir=Path.cwd())).resolve()
STATUS = {}
print('Working directory:', WORK)
print('Python:', sys.version.split()[0])
print('gcc:', shutil.which('gcc'), 'sbatch:', shutil.which('sbatch'))
print('Cluster measurements: NOT RUN until a batch report exists.')
def run(args, env=None, timeout=30):
    p = subprocess.run([str(a) for a in args], cwd=WORK, text=True,
                       capture_output=True, env=env, timeout=timeout)
    if p.returncode:
        raise RuntimeError(f'FAIL: command {args!r}, exit {p.returncode}\n{p.stdout}\n{p.stderr}')
    return p.stdout


## E1 · What does “nearest” mean?

**Real problem:** the service must return the correct item before we care how fast it runs.
**Prediction:** if each vector has unit length, how do we rank cosine similarity?
**Code:** cosine is the dot product of the two normalized vectors. We use float storage and double
accumulation. The highest score wins; an exactly equal computed score uses the smaller global ID.
IDs are zero-based. The reference uses the same generator and arithmetic precision, implemented independently in Python.

The next cell writes the complete C program. Inspect `vector`, the row-scoring loop and the result merge.
`mix` generates each component from its global row ID, so MPI partitions do not change the input.
This is deterministic data generation, not a scientific random-number generator.


In [ ]:
C_SOURCE = r"""/* M5S5 guided exact top-1 search. Deterministic synthetic vectors, not text embeddings.
 * Serial: gcc -O3 -std=c11 search.c -lm -o search_serial
 * OpenMP: gcc -O3 -std=c11 -fopenmp search.c -lm -o search_omp
 * Hybrid: mpicc -O3 -std=c11 -fopenmp -DUSE_MPI search.c -lm -o search_mpi
 */
#define _POSIX_C_SOURCE 200809L
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <math.h>
#include <time.h>
#ifdef _OPENMP
#include <omp.h>
#endif
#ifdef USE_MPI
#include <mpi.h>
#endif
static double now(void) { struct timespec t; clock_gettime(CLOCK_MONOTONIC,&t); return t.tv_sec+t.tv_nsec*1e-9; }
static void stop(const char *why) {
 fprintf(stderr,"FAIL: %s\n",why);
#ifdef USE_MPI
 MPI_Abort(MPI_COMM_WORLD,2);
#endif
 exit(2);
}
/* Stateless 32-bit hash: each rank can generate its own rows by global ID. */
static uint32_t mix(uint32_t x) { x^=x>>16; x*=0x7feb352dU; x^=x>>15; x*=0x846ca68bU; return x^(x>>16); }
static void vector(float *v,int d,uint32_t id) {
 double norm=0;
 for(int j=0;j<d;j++){ v[j]=(float)((int)(mix(id*747796405U+(uint32_t)j*2891336453U)>>8)%2001-1000)/1000.0f; norm+=(double)v[j]*v[j]; }
 norm=sqrt(norm); if(norm==0) stop("zero vector");
 for(int j=0;j<d;j++)v[j]=(float)(v[j]/norm);
}
static int number(const char *s,int hi) { char *end; long v=strtol(s,&end,10); if(*end||v<1||v>hi)stop("invalid size");return (int)v; }
int main(int argc,char **argv) {
 int rank=0,ranks=1,threads=1;
#ifdef USE_MPI
 int provided;MPI_Init_thread(&argc,&argv,MPI_THREAD_FUNNELED,&provided);
 if(provided<MPI_THREAD_FUNNELED)stop("MPI thread support too low");
 MPI_Comm_rank(MPI_COMM_WORLD,&rank);MPI_Comm_size(MPI_COMM_WORLD,&ranks);
#endif
 if(argc!=4)stop("usage: search N D Q (N<=1000000,D<=512,Q<=128)");
 int n=number(argv[1],1000000),d=number(argv[2],512),nq=number(argv[3],128);
 if(ranks>n)stop("more ranks than vectors");
#ifdef _OPENMP
 #pragma omp parallel
 {
  #pragma omp single
  threads=omp_get_num_threads();
 }
#endif
 int begin=(int)((long long)n*rank/ranks),end=(int)((long long)n*(rank+1)/ranks),local=end-begin;
 float *x=malloc((size_t)local*d*sizeof(float)),*q=malloc((size_t)nq*d*sizeof(float));
 double *scores=malloc((size_t)local*sizeof(double)),*best=malloc((size_t)nq*sizeof(double));
 int *ids=malloc((size_t)nq*sizeof(int));
 double *gather_scores=malloc((size_t)ranks*sizeof(double));int *gather_ids=malloc((size_t)ranks*sizeof(int));
 if(!x||!q||!scores||!best||!ids||!gather_scores||!gather_ids)stop("allocation failed");
 for(int i=0;i<local;i++)vector(x+(size_t)i*d,d,(uint32_t)(begin+i+1));
 for(int k=0;k<nq;k++)vector(q+(size_t)k*d,d,(uint32_t)(10000001+k));
 /* One untimed dot-product pass warms memory/code. It does not initialize results. */
 volatile double warm=0; for(int j=0;j<d;j++)warm+=(double)x[j]*q[j];(void)warm;
#ifdef USE_MPI
 MPI_Barrier(MPI_COMM_WORLD);
#endif
 double start=now(),compute=0,selection=0,collective=0;
 for(int k=0;k<nq;k++) {
  double t=now();
  /* Each thread writes a different scores[i]; no shared best-index race. */
  #pragma omp parallel for schedule(static)
  for(int i=0;i<local;i++) {
   double dot=0;
   for(int j=0;j<d;j++)dot+=(double)x[(size_t)i*d+j]*q[(size_t)k*d+j];
   scores[i]=dot;
  }
  compute+=now()-t;t=now();
  int id=begin;double value=scores[0];
  for(int i=1;i<local;i++)if(scores[i]>value||(scores[i]==value&&begin+i<id)){value=scores[i];id=begin+i;}
  selection+=now()-t;t=now();
#ifdef USE_MPI
  /* All MPI calls are outside OpenMP regions: FUNNELED is sufficient. */
  MPI_Gather(&value,1,MPI_DOUBLE,gather_scores,1,MPI_DOUBLE,0,MPI_COMM_WORLD);
  MPI_Gather(&id,1,MPI_INT,gather_ids,1,MPI_INT,0,MPI_COMM_WORLD);
  if(rank==0)for(int r=0;r<ranks;r++)if(gather_scores[r]>value||(gather_scores[r]==value&&gather_ids[r]<id)){value=gather_scores[r];id=gather_ids[r];}
#endif
  if(ranks>1) collective+=now()-t;
  best[k]=value;ids[k]=id;
 }
 double elapsed=now()-start;
#ifdef USE_MPI
 double times[4]={elapsed,compute,selection,collective},maxima[4];
 MPI_Reduce(times,maxima,4,MPI_DOUBLE,MPI_MAX,0,MPI_COMM_WORLD);
 elapsed=maxima[0];compute=maxima[1];selection=maxima[2];collective=maxima[3];
#endif
 if(rank==0){
  printf("{\"n\":%d,\"d\":%d,\"queries\":%d,\"ranks\":%d,\"threads\":%d,\"seconds\":%.9g,\"compute_max_s\":%.9g,\"selection_max_s\":%.9g,\"collective_max_s\":%.9g,\"hits\":[",n,d,nq,ranks,threads,elapsed,compute,selection,collective);
  for(int k=0;k<nq;k++)printf("%s{\"id\":%d,\"score\":%.17g}",k?",":"",ids[k],best[k]);
  puts("]}");
 }
 free(x);free(q);free(scores);free(best);free(ids);free(gather_scores);free(gather_ids);
#ifdef USE_MPI
 MPI_Finalize();
#endif
 return 0;
}
"""
(WORK/'search.c').write_text(C_SOURCE)
print(C_SOURCE)


In [ ]:
# Independent tiny reference: 17 rows, 7 dimensions, 3 queries. No benchmark here.
def f32(v): return struct.unpack('f', struct.pack('f', v))[0]
def mix(x):
    x &= 0xffffffff
    x ^= x >> 16; x = (x * 0x7feb352d) & 0xffffffff
    x ^= x >> 15; x = (x * 0x846ca68b) & 0xffffffff
    return x ^ (x >> 16)
def vector(identifier, dimensions):
    v = [f32(f32((mix(identifier*747796405+j*2891336453) >> 8) % 2001-1000)/f32(1000))
         for j in range(dimensions)]
    norm = math.sqrt(sum(x*x for x in v))
    return [f32(x/norm) for x in v]
def expected_hits(n, d, queries):
    data = [vector(i+1, d) for i in range(n)]
    hits = []
    for k in range(queries):
        query = vector(10000001+k, d)
        scores = [sum(x*y for x,y in zip(row,query)) for row in data]
        best = max(range(n), key=lambda i: (scores[i], -i))
        hits.append(dict(id=best, score=scores[best]))
    return hits
if not shutil.which('gcc'):
    STATUS['E1_tiny_reference'] = 'NOT RUN'
    print('NOT RUN: gcc is missing in this kernel. Load the site environment and rerun here or in a compute terminal.')
else:
    run(['gcc','-O3','-std=c11','search.c','-lm','-o','search_serial'])
    actual = json.loads(run([WORK/'search_serial',17,7,3]))
    expected = expected_hits(17,7,3)
    assert len(actual['hits']) == len(expected)
    for a,b in zip(actual['hits'],expected):
        assert a['id']==b['id'] and math.isclose(a['score'],b['score'],abs_tol=1e-10,rel_tol=1e-10)
    STATUS['E1_tiny_reference'] = 'PASS'
    print('PASS: C agrees with the independent Python tiny reference.')
    print(json.dumps(actual['hits'],indent=2))


**Result and explanation:** the printed IDs and scores are actual tiny-run outputs from your environment.
The time field from this fixture is not a useful performance benchmark.
**Student experiment:** change only the tiny row count from 17 to 19 in both reference and C call.
**Takeaway:** deterministic inputs and an independent check make errors easier to isolate.

**Checkpoint 1:** why does running without a crash not prove search correctness?
A. Every parallel program is wrong. B. It may return the wrong item. C. More threads prove correctness. D. Slurm checks the mathematics.
<details><summary>Expected reasoning</summary>B. Completion is necessary, but neither the scheduler nor thread count checks the returned neighbour. A is false.</details>


## E2 · Threads over one collection

**Real problem:** every stored vector must be compared with each query.
**Prediction:** will four threads be four times faster? Which data can they share safely?
**Code:** `#pragma omp parallel for schedule(static)` splits row scoring. Each iteration writes its own
`scores[i]`; input vectors and the query are read-only. A serial scan then selects the local winner.
That scan is intentionally simple and can limit performance. Queries are processed sequentially in this guided version.

We compare a separately compiled serial baseline, OpenMP with one thread, then two and four threads.
Each configuration runs three times in one bounded allocation. We retain all repetitions and use medians.
Generation, normalization, allocation, compilation, launch and correctness comparison are outside the C timing region.
The timed region includes scoring, local selection and, for MPI, the per-query collective/merge.
This is **search-phase throughput**, not end-to-end request latency.


In [ ]:
FILES = {'container.sbatch': '#!/bin/bash -l\n#SBATCH --job-name=m5s5-container\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=4\n#SBATCH --mem=1G\n#SBATCH --time=00:06:00\n#SBATCH --output=m5s5-container-%j.out\nset -euo pipefail\ncd "$SLURM_SUBMIT_DIR"\nsource ./site_env.sh\nprintf \'JOB_ID=%s HOST=%s DATE=%s\\n\' "$SLURM_JOB_ID" "$(hostname)" "$(date -Is)"\nmodule -t list 2>&1 || true\nlscpu | head -25 || true\npython3 run_experiments.py container --n "${M5S5_N:-65536}" --queries "${M5S5_QUERIES:-16}"\n', 'hybrid.sbatch': '#!/bin/bash -l\n#SBATCH --job-name=m5s5-hybrid\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=2\n#SBATCH --cpus-per-task=2\n#SBATCH --mem=1G\n#SBATCH --time=00:06:00\n#SBATCH --output=m5s5-hybrid-%j.out\nset -euo pipefail\ncd "$SLURM_SUBMIT_DIR"\nsource ./site_env.sh\nprintf \'JOB_ID=%s HOST=%s DATE=%s\\n\' "$SLURM_JOB_ID" "$(hostname)" "$(date -Is)"\nmodule -t list 2>&1 || true\nlscpu | head -25 || true\npython3 run_experiments.py hybrid --n "${M5S5_N:-65536}" --queries "${M5S5_QUERIES:-16}"\n', 'cpu.sbatch': '#!/bin/bash -l\n#SBATCH --job-name=m5s5-cpu\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=4\n#SBATCH --mem=1G\n#SBATCH --time=00:06:00\n#SBATCH --output=m5s5-cpu-%j.out\nset -euo pipefail\ncd "$SLURM_SUBMIT_DIR"\nsource ./site_env.sh\nprintf \'JOB_ID=%s HOST=%s DATE=%s\\n\' "$SLURM_JOB_ID" "$(hostname)" "$(date -Is)"\nmodule -t list 2>&1 || true\nlscpu | head -25 || true\npython3 run_experiments.py cpu --n "${M5S5_N:-65536}" --queries "${M5S5_QUERIES:-16}"\n', 'run_experiments.py': '"""Bounded guided experiments. Significant work requires a Slurm batch allocation.\n--local is for instructor validation on a local workstation, never a login node.\n"""\nfrom pathlib import Path\nimport argparse, hashlib, json, math, os, shutil, socket, subprocess, time, traceback\n\nROOT = Path(__file__).resolve().parent\n\ndef command(args, env=None, timeout=180):\n    p = subprocess.run([str(a) for a in args], text=True, capture_output=True,\n                       timeout=timeout, env=env)\n    if p.returncode:\n        raise RuntimeError(f\'{args!r}\\nexit={p.returncode}\\n{p.stdout}\\n{p.stderr}\')\n    return p.stdout\n\ndef equivalent(actual, reference):\n    assert (actual[\'n\'], actual[\'d\'], actual[\'queries\']) == (reference[\'n\'], reference[\'d\'], reference[\'queries\'])\n    assert len(actual[\'hits\']) == len(reference[\'hits\'])\n    for a, b in zip(actual[\'hits\'], reference[\'hits\']):\n        assert a[\'id\'] == b[\'id\'], (a, b)\n        assert math.isfinite(a[\'score\']) and math.isclose(a[\'score\'], b[\'score\'], rel_tol=1e-10, abs_tol=1e-10)\n    assert math.isfinite(actual[\'seconds\']) and actual[\'seconds\'] > 0\n\ndef main():\n    os.chdir(ROOT)\n    p = argparse.ArgumentParser()\n    p.add_argument(\'mode\', choices=[\'cpu\', \'hybrid\', \'container\'])\n    p.add_argument(\'--local\', action=\'store_true\')\n    p.add_argument(\'--n\', type=int, default=65536)\n    p.add_argument(\'--d\', type=int, default=128)\n    p.add_argument(\'--queries\', type=int, default=16)\n    a = p.parse_args()\n    if not a.local and not os.environ.get(\'SLURM_JOB_ID\'):\n        raise RuntimeError(\'NOT RUN: submit the supplied batch script; do not benchmark on a login/Jupyter node.\')\n    assert 1 <= a.n <= 1000000 and 1 <= a.d <= 512 and 1 <= a.queries <= 128\n    assert a.n*a.d*4 <= 256*1024*1024, \'Keep vector storage at or below 256 MiB for the 1 GB allocation\'\n    out = ROOT / \'results\'; out.mkdir(exist_ok=True)\n    label = f\'{a.mode}-{os.environ.get("SLURM_JOB_ID", "local")}-{time.time_ns()}\'\n    records = []\n    meta = dict(mode=a.mode, execution=\'LOCAL\' if a.local else \'SLURM\', job_id=os.environ.get(\'SLURM_JOB_ID\'),\n                host=socket.gethostname(), source_sha256=hashlib.sha256((ROOT/\'search.c\').read_bytes()).hexdigest(),\n                configuration=dict(n=a.n, d=a.d, queries=a.queries),\n                allocation={k: os.environ.get(k) for k in [\'SLURM_JOB_NUM_NODES\',\'SLURM_NTASKS\',\'SLURM_CPUS_PER_TASK\']})\n    try:\n        cc = shutil.which(\'gcc\')\n        if not cc: raise RuntimeError(\'gcc missing: load the site GCC module in site_env.sh\')\n        meta[\'compiler\'] = command([cc, \'--version\']).splitlines()[0]\n        meta[\'flags\'] = \'-O3 -std=c11; OpenMP adds -fopenmp; MPI adds -DUSE_MPI\'\n        command([cc, \'-O3\',\'-std=c11\',\'search.c\',\'-lm\',\'-o\',\'search_serial\'])\n        size = [str(a.n), str(a.d), str(a.queries)]\n        def run(exe, ranks=1, threads=1, prefix=None):\n            env = os.environ.copy()\n            env.update(OMP_NUM_THREADS=str(threads), OMP_DYNAMIC=\'FALSE\', OMP_PLACES=\'cores\', OMP_PROC_BIND=\'close\')\n            cmd = (prefix or []) + [str(exe)] + size\n            if not a.local:\n                cmd = [\'srun\',\'--exact\',\'--exclusive\',\'--nodes=1\',f\'--ntasks={ranks}\',f\'--cpus-per-task={threads}\',\'--cpu-bind=cores\'] + cmd\n            elif ranks > 1:\n                launcher = shutil.which(\'mpiexec\')\n                if not launcher: raise RuntimeError(\'local MPI launcher missing\')\n                cmd = [launcher, \'-n\', str(ranks)] + cmd\n            result = json.loads(command(cmd, env))\n            assert result[\'ranks\']==ranks and result[\'threads\']==threads, \'Runtime did not use the requested team size\'\n            return result, cmd\n        reference, _ = run(ROOT/\'search_serial\')\n        if a.mode == \'cpu\':\n            command([cc,\'-O3\',\'-std=c11\',\'-fopenmp\',\'search.c\',\'-lm\',\'-o\',\'search_omp\'])\n            configurations = [(\'serial\',ROOT/\'search_serial\',1,1,None)] + [(f\'omp-{t}\',ROOT/\'search_omp\',1,t,None) for t in [1,2,4]]\n        elif a.mode == \'hybrid\':\n            mpi = shutil.which(\'mpicc\')\n            if not mpi:\n                meta.update(status=\'SKIPPED\',reason=\'mpicc unavailable: load a site-compatible MPI module; CPU measurements remain usable\')\n                return\n            meta[\'mpi_wrapper\'] = command([mpi,\'--version\']).splitlines()[0]\n            command([mpi,\'-O3\',\'-std=c11\',\'-fopenmp\',\'-DUSE_MPI\',\'search.c\',\'-lm\',\'-o\',\'search_mpi\'])\n            configurations = [(\'mpi-2x2\',ROOT/\'search_mpi\',2,2,None)]\n        else:\n            image = Path(os.environ.get(\'M5S5_IMAGE\',\'/nonexistent\')).expanduser()\n            app = shutil.which(\'apptainer\')\n            if not app or not image.is_file():\n                meta.update(status=\'SKIPPED\',reason=\'Set M5S5_IMAGE to an existing trusted C/OpenMP-capable SIF and load Apptainer\')\n                return\n            meta[\'image_path\'] = str(image.resolve())\n            h = hashlib.sha256()\n            with image.open(\'rb\') as f:\n                for block in iter(lambda:f.read(1024*1024), b\'\'): h.update(block)\n            meta[\'image_sha256\'] = h.hexdigest()\n            prefix = [app,\'exec\',\'--cleanenv\',\'--bind\',str(ROOT)+\':/work\',\'--pwd\',\'/work\',str(image.resolve())]\n            # Compile inside the image, rather than assuming host libgomp/glibc compatibility.\n            build = prefix + [\'gcc\',\'-O3\',\'-std=c11\',\'-fopenmp\',\'/work/search.c\',\'-lm\',\'-o\',\'/work/search_container\']\n            if not a.local: build = [\'srun\',\'--exact\',\'-N1\',\'-n1\',\'-c1\'] + build\n            probe = subprocess.run(build,text=True,capture_output=True,timeout=60)\n            if probe.returncode:\n                meta.update(status=\'SKIPPED\',reason=\'Image lacks a compatible gcc/OpenMP build environment\',diagnostic=probe.stderr[-3000:])\n                return\n            os.environ[\'APPTAINERENV_OMP_NUM_THREADS\']=\'4\'\n            os.environ[\'APPTAINERENV_OMP_DYNAMIC\']=\'FALSE\'\n            os.environ[\'APPTAINERENV_OMP_PLACES\']=\'cores\'\n            os.environ[\'APPTAINERENV_OMP_PROC_BIND\']=\'close\'\n            configurations = [(\'container-omp-4\',\'/work/search_container\',1,4,prefix)]\n        for name, exe, ranks, threads, prefix in configurations:\n            for repeat in range(3):\n                result, cmd = run(exe,ranks,threads,prefix)\n                equivalent(result,reference)\n                record = dict(configuration=name,repeat=repeat+1,status=\'PASS\',command=cmd,**result)\n                records.append(record)\n                print(json.dumps(record),flush=True)\n        meta[\'status\']=\'PASS\'\n    except Exception as e:\n        meta.update(status=\'FAIL\',reason=str(e),traceback=traceback.format_exc())\n        raise\n    finally:\n        meta[\'records\']=records\n        (out/(label+\'.json\')).write_text(json.dumps(meta,indent=2)+\'\\n\')\n        print(\'RESULT\',meta[\'status\'],out/(label+\'.json\'),meta.get(\'reason\',\'\'),flush=True)\n\nif __name__==\'__main__\':main()\n', 'site_env.sh': '# Add the exact module load commands used on SciTech here.\n# Check module avail / module list first. Do not install MPI or change the cluster.\n# Example format only: module load <site-GCC-module> <matching-MPI-module>\n# CPU needs Python 3 and gcc with OpenMP. Hybrid also needs matching mpicc/srun.\n# Container additionally needs Apptainer and an existing compatible image:\n# export M5S5_IMAGE=/absolute/path/to/site-approved-image.sif\n'}
for name, text in FILES.items():
    (WORK/name).write_text(text)
print('Created:', ', '.join(FILES))
print((WORK/'cpu.sbatch').read_text())
print('EDIT site_env.sh before submission. Files generated, jobs NOT RUN.')


In [ ]:
# Review WORK/site_env.sh first. Set a flag only for a new intended submission.
SUBMIT_CPU = False
SUBMIT_HYBRID = False
SUBMIT_CONTAINER = False
for mode, enabled in [('cpu',SUBMIT_CPU),('hybrid',SUBMIT_HYBRID),('container',SUBMIT_CONTAINER)]:
    if not enabled:
        print(mode, 'NOT RUN by this cell; use an existing report if already submitted.')
    elif not shutil.which('sbatch'):
        print(mode, 'NOT RUN: sbatch missing here. Submit from the cluster terminal.')
    else:
        print(mode, 'submitted job', run(['sbatch','--parsable',mode+'.sbatch']).strip())
print('Submission is not PASS. Wait for a completed report, then run the analysis cells.')


**Run from a cluster terminal instead:** `cd` into the printed working directory, edit `site_env.sh`,
then `sbatch cpu.sbatch`. Check `squeue -u $USER`; inspect `m5s5-cpu-JOBID.out` and `results/*.json`.
Use `sacct -j JOBID --format=JobID,State,ExitCode,Elapsed,AllocCPUS` for scheduler evidence.
If a job fails before Python starts, no JSON may exist: the Slurm log and nonzero exit code are failure evidence.
Cancel unwanted jobs with `scancel JOBID`. This notebook does not wait indefinitely or resubmit automatically.

**Student experiment:** change only the query count from 16 to 8:
`sbatch --export=ALL,M5S5_QUERIES=8 cpu.sbatch`.
Compare throughput and elapsed time, but do not mix these timings into the fixed-workload speedup table.
**Takeaway:** configuration, numerical result and timing region must agree before comparing speed.


## E3 · MPI shards and threads inside each shard

**Real problem:** a larger collection may need to be partitioned across processes.
**Prediction:** if one rank finds the best item in its shard, is that necessarily the global answer?
**Code:** rank r owns `[N*r/P, N*(r+1)/P)`. Each rank stores only that interval plus the small query set.
Two `MPI_Gather` calls send one score and one ID per rank per query to rank 0, which merges the winners.
MPI calls happen outside threaded regions and request `MPI_THREAD_FUNNELED`.

**Run:** after confirming a compatible MPI toolchain and site launch configuration, submit `sbatch hybrid.sbatch`.
It uses **2 ranks × 2 threads = 4 CPUs**, on one node. Compare with the CPU job's 1 rank × 4 threads.
Both jobs request four CPUs; different allocations may land on different hardware, so compare host/CPU records.
This demonstrates a hybrid layout, not inter-node scaling. Multi-node work is not required here.

**Result and explanation:** each completed repetition checks every query's winning ID and score against
a fresh serial reference for the same input. Missing MPI is explicitly SKIPPED. A bad launcher or wrong answer is FAIL.
Record diagnostics; do not install a new MPI stack or guess process-manager options.
**Student experiment:** predict how many winner records rank 0 receives per query if ranks change from 2 to 4.
**Takeaway:** local correctness needs a correct global merge.

**Checkpoint 2:** which is a fair fixed-CPU comparison?
A. 4 ranks × 4 threads versus 1 × 4. B. Change dataset and rank count together.
C. 2 × 2 versus 1 × 4, same workload and comparable hardware. D. Compare queue waiting times.
<details><summary>Expected reasoning</summary>C. A changes the CPU budget, B changes work and D measures scheduling delay rather than search performance.</details>


## E4 · Same search in an existing container

**Real problem:** another environment may have different compiler and runtime libraries.
**Prediction:** does a SIF guarantee identical timing or correct MPI integration?
**Code / Run:** use an existing trusted, architecture-compatible **gcc + OpenMP** image available on SciTech.
Set its absolute path in `site_env.sh`: `export M5S5_IMAGE=/path/to/existing.sif`, then submit `container.sbatch`.
Reuse the image inventory from M5S3 or ask the instructor for an approved path; no image path is invented here.
The runner compiles this C source **inside** the image, uses `--cleanenv`, binds the owned folder at `/work`,
sets the OpenMP environment explicitly, and executes four threads within Slurm.

**Result / Explanation:** it checks output against the native serial reference and records the SIF SHA256.
A Python-only image without gcc/OpenMP is incompatible with this extension and is SKIPPED with a diagnostic.
An incorrect numerical result is FAIL. No downloads or installations are performed.
The checksum records image identity, not provenance. The host kernel, architecture, placement and hardware
still affect execution; containerized MPI requires additional site compatibility checks and is not attempted here.

**Student experiment:** inspect the launch command and identify the host folder, container folder,
thread count and compiler environment. Which would another student need to reproduce your run?
**Takeaway:** preserve the application environment and record the host execution context.


## E5 · Read the evidence, not an expected speedup

**Real problem:** timings from different problem sizes or failed jobs cannot be mixed.
**Prediction:** if four threads are slower, has the experiment necessarily failed?
**Code / Run:** load the latest report per mode for the default 65536 × 128 collection and 16 queries,
with the current source hash. All tables below use actual records from your files. There is no fallback timing dataset.
If a job is still queued or failed before producing a report, inspect its scheduler state and log.


In [ ]:
DEFAULT = dict(n=65536,d=128,queries=16)
source_hash = hashlib.sha256((WORK/'search.c').read_bytes()).hexdigest()
reports = {}
for path in sorted((WORK/'results').glob('*.json'), key=lambda p:p.stat().st_mtime_ns) if (WORK/'results').exists() else []:
    r = json.loads(path.read_text())
    if r.get('source_sha256')==source_hash and r.get('configuration')==DEFAULT:
        reports[r['mode']] = r
summary=[]
for mode in ['cpu','hybrid','container']:
    report=reports.get(mode)
    STATUS[mode] = report['status'] if report else 'NOT RUN'
    print(mode, STATUS[mode], report.get('reason','') if report else '')
    if not report or report['status']!='PASS': continue
    print('Evidence:',report['execution'],'job',report['job_id'],'host',report['host'])
    groups={}
    for row in report['records']: groups.setdefault(row['configuration'],[]).append(row)
    for name, values in groups.items():
        assert len(values)==3 and all(v['status']=='PASS' for v in values)
        t=median(v['seconds'] for v in values)
        summary.append(dict(configuration=name,median_s=t,min_s=min(v['seconds'] for v in values),
                            max_s=max(v['seconds'] for v in values),queries_per_s=DEFAULT['queries']/t,
                            cpus=values[0]['ranks']*values[0]['threads'],execution=report['execution'],
                            job_id=report['job_id'],host=report['host']))
baseline=next((s['median_s'] for s in summary if s['configuration']=='serial'),None)
for row in summary:
    row['speedup']=baseline/row['median_s'] if baseline else None
    row['efficiency']=row['speedup']/row['cpus'] if baseline else None
    print(row)
if summary:
    with (WORK/'summary.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(summary[0]));writer.writeheader();writer.writerows(summary)
else: print('NOT RUN: no matching completed measurements. No performance claim can be made.')


In [ ]:
if summary:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print('SKIPPED: plotting library unavailable; use the printed table and summary.csv. Do not install it for this lab.')
    else:
        fig, ax=plt.subplots(figsize=(8,4))
        ax.bar([s['configuration'] for s in summary],[s['median_s'] for s in summary],color='#2F5597')
        ax.set_ylabel('Median search-phase elapsed time (s)')
        ax.set_title('Actual execution records: fixed workload, 3 repetitions')
        ax.tick_params(axis='x',rotation=20);fig.tight_layout()
        fig.savefig(WORK/'measured_times.png',dpi=160);plt.show()
else: print('NOT RUN: plot needs real measurement records.')


**Result / explanation:** use the range of repetitions as well as the median. A short search may not
show useful scaling. Thread overhead, a serial selection scan, memory traffic and repeated collectives
are hypotheses, not conclusions until supported by measurements.

The raw records contain `compute_max_s`, `selection_max_s` and `collective_max_s`.
For MPI each is a maximum across ranks; these maxima may belong to different ranks and **must not
be added as a mutually exclusive runtime breakdown**. Collective time includes waiting and root merge,
not just network transfer. Allocation, initialization and input generation are outside these phase timings.

**Student experiment:** inspect one raw CPU and hybrid report. Propose one change to test the dominant
cost while keeping the input and correctness rule fixed. Do not claim memory-bandwidth saturation solely
because scaling flattened. A larger query count is a sensitivity experiment, not fixed-workload speedup.
**Takeaway:** a defensible result states what ran, how it was checked and what the measurements support.

**Checkpoint 3:** the container returns identical neighbours but is slower. Best conclusion?
A. Containers always slow applications. B. The result is invalid. C. Record this result and investigate
compiler/runtime/placement differences. D. Replace the measured timing with the native one.
<details><summary>Expected reasoning</summary>C. One comparison does not establish a universal performance effect. Numerical agreement and speed are different questions.</details>

## Exit and handover
Keep `search.c`, `run_experiments.py`, `site_env.sh`, batch files, source/image hashes, raw JSON, Slurm
logs and summary CSV. Record the repository commit used to obtain this notebook and exact commands.
Record any local edits with their own source hash. Never label workstation results as SciTech measurements.

The separate individual assignment requires **batched top-k retrieval**, an independently justified
optimization and your own evidence. The group milestone uses your group's chosen project, not this
guided output. Course assessment and AI-use rules continue to apply.

### Instructor validation boundary
The released source and all notebook cells were tested locally. Serial/OpenMP correctness and bounded
CPU runs passed; MPI, Slurm and Apptainer were unavailable locally and remain **NOT RUN on SciTech**.
Compiler module selection, MPI launch and image compatibility still need a real site validation run.
Published notebook outputs are cleared to prevent confusing instructor-local timings with student measurements.

Primary teaching references: official syllabus Sessions 21–25 (pp.15–17); repository cluster guide;
M5S3 environment/container notebook; M4S5 asynchronous notebook conventions. No new theory reading is required.
